# 14 — Preparar a avaliação temporal da referência

Autor: Wanderson Ferreira. Preparação: 2026-10-09.

**HGB = Histogram-based Gradient Boosting**, referência já congelada e treinada
em abril. Esta etapa parte de `main` após o merge do PR #1. O objetivo é fixar a
pergunta e os limites estatísticos antes de abrir setembro. Não executamos aqui
a avaliação do modelo nem o streaming.


## Pergunta e alternativas

Como a referência de abril ordena as transações e prioriza clientes de 02–15/09?
Optuna conservou a referência: nenhum dos 20 trials passou o gate. Não escolhemos
retroativamente o maior AP como candidato confirmatório, nem reutilizamos o teste
de maio como confirmação independente de hipóteses posteriores.

Também não adotamos bootstrap de transações independentes: clientes, terminais e
históricos se repetem. Os 14 dias de contexto das features decorrem da causalidade
e do atraso de rótulos; não justificam blocos de reamostragem ou potência. Uma
inferência sobre períodos futuros exige condições adicionais, declaradas antes
que os resultados reservados orientem novas escolhas.


## Decisão e fronteiras

O snapshot `reference_assessment_v1` vincula os recibos de congelamento,
desenvolvimento e seleção. O plano lê somente quatro JSONs e confere hashes;
`--require-committed` também exige código e inputs rastreados, sem alterações.
Mesmo um plano commitado continua sendo preparação, sem executor de avaliação.

| Janela de 2018 | Finalidade |
| --- | --- |
| 19/08–01/09 | Contexto causal da avaliação, sem treino |
| 02–15/09 | Avaliar o modelo fixo e relatar os 14 dias |
| 16–30/09 | Reserva de replay operacional; outro contrato de eventos |

A fila usa máximo score por cliente/dia e até 100 clientes. Essa política é
retrospectiva por dia completo. Streaming precisará definir quando emitir alertas,
tratar estado causal e entregar o feedback sete dias depois no relógio dos eventos.


In [ ]:
# Apenas metadados do checkout; sem acesso a dados/modelos nativos.
from fraud_detection_mlops.modeling.experiments.reference_temporal_assessment import (
    prepare_assessment_plan,
)

plan = prepare_assessment_plan()
assert plan["current_authorization"]["native_data_reads"] is False
assert plan["model"]["loaded"] is False
{key: value["days"] for key, value in plan["windows"].items()}

## Referência condicional de fila aleatória

Uma política hipotética sorteia clientes uniformemente, sem reposição, com
sorteios novos e independentes por dia. Condicionando nas populações e rótulos
observados, calculamos capturas esperadas, intervalo central de capturas e
probabilidade de capturar pelo menos o total observado. A convolução usa
log-probabilidades para preservar caudas pequenas.

A independência pertence aos sorteios da política, não às entidades observadas.
O intervalo não é um intervalo de confiança do HGB e a cauda não testa qualidade
futura. O exemplo abaixo é inventado, sem qualquer métrica nativa do projeto.


In [ ]:
import pandas as pd

from fraud_detection_mlops.evaluation.conditional_queue_reference import (
    uniform_queue_reference,
)

toy_days = pd.DataFrame(
    [
        {
            "date": "2000-01-01",
            "customers": 5,
            "alerts": 2,
            "fraudulent_customers": 2,
            "fraudulent_customers_in_alerts": 2,
        },
        {
            "date": "2000-01-02",
            "customers": 4,
            "alerts": 2,
            "fraudulent_customers": 1,
            "fraudulent_customers_in_alerts": 1,
        },
    ]
)
uniform_queue_reference(toy_days, alert_budget=2)

## Conferências e limitações

Os 22 novos casos sintéticos verificam hashes, ausência de leitura de dados,
inputs/código commitados e rejeição de execução reservada pela CLI. O cálculo
condicional é comparado com enumeração independente de todas as filas possíveis
em populações pequenas, incluindo clientes recorrentes, além de uma cauda extrema
com resultado em log10. O recibo abaixo registra a validação completa do projeto.

Nenhum dado nativo de setembro foi lido; o modelo real não foi carregado. Os fits
e runs dos testes usam fixtures temporárias; não houve fit ou run nativos nem replay.
O relato de Optuna foi vinculado como relato do autor,
sem substituir verificação do banco nativo. Inferência confirmatória temporal,
exportação real, Docker e orquestração continuam com entregas próprias.

Próximo passo: executor auditável para verificar dados/modelo, construir features
causais, preservar cobertura diária e publicar resultados verificáveis da janela
fixada. Uma vez avaliada, essa janela não permanecerá independente para novas
hipóteses orientadas pelos seus resultados.

[Documento principal](../../docs/modeling/EVALUATION_PROTOCOL.md#protocolo-estatístico-da-referência--v1),
[recibo](../../references/evidence/reference_assessment_preparation_2026-10-09.json),
[mural](../../docs/project/ROADMAP.md#próxima-entrega-concreta).
